# 06 — Part 3: LLM Correction Experiment
**IS5126 LLM-Enhanced Credit Risk Assessment — NUS**

This notebook covers **Part 3**: can Qwen correct ML mistakes on borderline cases?

**Setup:**
- Sample 200 loans from the **full LC test set** (no desc requirement)
- Run trained primary ML model → get predictions + confidence scores
- Define **borderline zone**: ML predicted probability ∈ [0.3, 0.7]
- Send borderline cases to Qwen for a second-pass review
- Evaluate: does LLM correction improve accuracy on these hard cases?

**Expected conclusion:** LLM second-pass meaningfully improves accuracy on borderline cases, demonstrating complementary value to traditional ML.

## 0. Setup

In [ ]:
!pip install -q openai xgboost tqdm

In [ ]:
import os, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import joblib
from tqdm.auto import tqdm
from openai import OpenAI

from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    f1_score, confusion_matrix, roc_auc_score, roc_curve
)

warnings.filterwarnings('ignore')

COLORS = {
    'primary': '#1F4E79', 'accent': '#2E86AB',
    'good': '#28A745', 'bad': '#DC3545',
    'qwen': '#E74C3C', 'neutral': '#6C757D',
}

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

DRIVE_DIR  = '/content/drive/MyDrive/is5126'
DATA_DIR   = f'{DRIVE_DIR}/data/processed'
MODEL_DIR  = f'{DRIVE_DIR}/models'

RANDOM_STATE    = 42
TARGET_COL      = 'default'
SAMPLE_N        = 200
BORDERLINE_N    = 100   # how many of 200 are borderline cases
BORDERLINE_LOW  = 0.3
BORDERLINE_HIGH = 0.7
QWEN_MODEL      = 'qwen3-max'
QWEN_DELAY      = 0.5

SAMPLE_200_PATH  = f'{DATA_DIR}/sample_200.parquet'
QWEN_CACHE_PATH  = f'{MODEL_DIR}/qwen_cache_200.json'

## 1. Load Test Set & Primary ML Model

In [ ]:
df_test_full = pd.read_parquet(f'{DATA_DIR}/test.parquet')

# Load primary model (XGB, no grade, optimized)
model_path = f'{MODEL_DIR}/XGB_B_no_grade_optimized.joblib'
if not os.path.exists(model_path):
    model_path = f'{MODEL_DIR}/XGB_B_no_grade.joblib'
primary_model = joblib.load(model_path)
print(f'Loaded model: {model_path.split("/")[-1]}')

# Load feature metadata
with open(f'{DATA_DIR}/feature_metadata.json') as f:
    feat_meta = json.load(f)

grade_related = feat_meta.get('grade_related', ['grade_num', 'sub_grade_num', 'int_rate'])
feature_cols = [c for c in feat_meta['feature_cols']
                if c not in grade_related and c in df_test_full.columns]

print(f'Test set: {len(df_test_full):,} rows')
print(f'Features: {len(feature_cols)}')

In [ ]:
# Get ML predictions on full test set
X_test_full = df_test_full[feature_cols].fillna(0)
y_test_full = df_test_full[TARGET_COL]

# Try loading pre-saved proba first
proba_path = f'{MODEL_DIR}/primary_test_proba.npy'
if os.path.exists(proba_path) and len(np.load(proba_path)) == len(df_test_full):
    ml_proba = np.load(proba_path)
    print('Loaded saved proba.')
else:
    ml_proba = primary_model.predict_proba(X_test_full)[:, 1]
    print('Re-computed proba.')

df_test_full = df_test_full.copy()
df_test_full['ml_proba'] = ml_proba
df_test_full['is_borderline'] = (
    (ml_proba >= BORDERLINE_LOW) & (ml_proba <= BORDERLINE_HIGH)
)

borderline_count = df_test_full['is_borderline'].sum()
print(f'Borderline cases in test set: {borderline_count:,} ({borderline_count/len(df_test_full)*100:.1f}%)')

## 2. Stratified Sample of 200

In [ ]:
if os.path.exists(SAMPLE_200_PATH):
    print('Loading cached 200 sample...')
    df_200 = pd.read_parquet(SAMPLE_200_PATH)
else:
    np.random.seed(RANDOM_STATE)

    # 100 borderline + 100 confident (50/50 true label each group)
    df_border = df_test_full[df_test_full['is_borderline']]
    df_confident = df_test_full[~df_test_full['is_borderline']]

    border_sample = df_border.groupby(TARGET_COL, group_keys=False).apply(
        lambda x: x.sample(n=min(BORDERLINE_N // 2, len(x)), random_state=RANDOM_STATE)
    ).head(BORDERLINE_N)

    confident_n = SAMPLE_N - len(border_sample)
    confident_sample = df_confident.groupby(TARGET_COL, group_keys=False).apply(
        lambda x: x.sample(n=min(confident_n // 2, len(x)), random_state=RANDOM_STATE)
    ).head(confident_n)

    df_200 = pd.concat([border_sample, confident_sample]).reset_index(drop=True)
    df_200.to_parquet(SAMPLE_200_PATH, index=False)
    print(f'Saved 200 sample.')

print(f'200 sample shape: {df_200.shape}')
print(f'Borderline: {df_200["is_borderline"].sum()} | Confident: {(~df_200["is_borderline"]).sum()}')
print(f'Default rate (overall): {df_200[TARGET_COL].mean()*100:.1f}%')
print(f'Default rate (borderline): {df_200[df_200["is_borderline"]][TARGET_COL].mean()*100:.1f}%')

In [ ]:
# ML predictions on 200 sample
X_200 = df_200[feature_cols].fillna(0)
y_200 = df_200[TARGET_COL]
ml_proba_200 = df_200['ml_proba'].values

# KS-optimal threshold from full test set
fpr_full, tpr_full, thr_full = roc_curve(y_test_full, ml_proba)
ml_threshold = thr_full[np.argmax(tpr_full - fpr_full)]
ml_pred_200 = (ml_proba_200 >= ml_threshold).astype(int)

ml_acc      = accuracy_score(y_200, ml_pred_200)
ml_bacc     = balanced_accuracy_score(y_200, ml_pred_200)
ml_f1       = f1_score(y_200, ml_pred_200, zero_division=0)

# Split by borderline/confident
mask_border = df_200['is_borderline'].values
ml_acc_border = accuracy_score(y_200[mask_border], ml_pred_200[mask_border])
ml_acc_conf   = accuracy_score(y_200[~mask_border], ml_pred_200[~mask_border])

print('=== ML Model on 200 Sample ===')
print(f'  Overall accuracy:     {ml_acc:.4f}')
print(f'  Balanced accuracy:    {ml_bacc:.4f}')
print(f'  F1:                   {ml_f1:.4f}')
print(f'  Borderline accuracy:  {ml_acc_border:.4f} (n={mask_border.sum()})')
print(f'  Confident accuracy:   {ml_acc_conf:.4f}  (n={(~mask_border).sum()})')
print(f'  Threshold used:       {ml_threshold:.4f}')

## 3. Qwen Second-Pass Review

For each borderline case, we serialize the full applicant profile (all features) into natural language and ask Qwen to make a binary DEFAULT / NO DEFAULT decision.

In [ ]:
# Compute SHAP values for df_200 and save shap_context_200.json
# This runs fast in Colab (200 rows). Upload the output to Drive.
import shap

SHAP_CONTEXT_PATH = f'{DATA_DIR}/shap_context_200.json'

if os.path.exists(SHAP_CONTEXT_PATH):
    print(f'SHAP context already exists: {SHAP_CONTEXT_PATH}')
    with open(SHAP_CONTEXT_PATH) as f:
        shap_context = json.load(f)
else:
    print('Computing SHAP values for 200 samples...')
    explainer = shap.TreeExplainer(primary_model)
    X_200_for_shap = X_200[feature_cols].fillna(0) if 'feature_cols' in dir() else X_200
    shap_values = explainer.shap_values(X_200_for_shap)

    # Use shap_values[:, :, 1] for binary classifiers (class=1 = default)
    if isinstance(shap_values, list):
        sv = shap_values[1]   # class 1 (default)
    elif shap_values.ndim == 3:
        sv = shap_values[:, :, 1]
    else:
        sv = shap_values

    feat_names = X_200_for_shap.columns.tolist()
    shap_context = {}
    for i, row_idx in enumerate(df_200.index):
        row_shap = sv[i]
        # Top 3 risk factors (positive SHAP = pushes toward default)
        top_risk_idx  = row_shap.argsort()[-3:][::-1]
        top_prot_idx  = row_shap.argsort()[:2]
        risk_factors  = [feat_names[j] for j in top_risk_idx if row_shap[j] > 0]
        prot_factors  = [feat_names[j] for j in top_prot_idx if row_shap[j] < 0]
        shap_context[str(row_idx)] = {
            'risk_factors':      risk_factors,
            'protective_factors': prot_factors,
        }

    with open(SHAP_CONTEXT_PATH, 'w') as f:
        json.dump(shap_context, f)
    print(f'Saved SHAP context for {len(shap_context)} rows to {SHAP_CONTEXT_PATH}')
    print('Upload this file to Drive, then copy to SOC ~/is5126/data/ for the correction script.')


In [ ]:
# Feature display names for readable profile serialization
FEATURE_LABELS = {
    'loan_amnt':             'Loan amount requested',
    'annual_inc':            'Annual income (self-reported)',
    'dti':                   'Debt-to-income ratio (%)',
    'fico_score':            'FICO credit score',
    'revol_util':            'Credit utilization rate (%)',
    'revol_bal':             'Revolving balance',
    'installment':           'Monthly installment',
    'open_acc':              'Number of open credit lines',
    'delinq_2yrs':           'Delinquencies in past 2 years',
    'inq_last_6mths':        'Credit inquiries in last 6 months',
    'credit_history_months': 'Credit history length (months)',
    'is_long_term':          'Loan term (1=60 months, 0=36 months)',
    'loan_to_income':        'Loan-to-income ratio',
    'installment_to_income': 'Monthly payment-to-income ratio',
    'purpose':               'Loan purpose',
    'home_ownership':        'Home ownership status',
    'emp_length_num':        'Employment length (years)',
    'verification_status':   'Income verification status',
}

def serialize_profile(row):
    """Convert a loan application row to readable natural language."""
    lines = ['Loan Applicant Profile:']
    for col, label in FEATURE_LABELS.items():
        if col in row.index and pd.notna(row[col]):
            val = row[col]
            if col in ['loan_amnt', 'annual_inc', 'revol_bal', 'installment']:
                val = f'${val:,.0f}'
            elif col in ['dti', 'revol_util', 'loan_to_income', 'installment_to_income']:
                val = f'{val:.2f}%'
            elif col == 'fico_score':
                val = f'{val:.0f}'
            elif col == 'is_long_term':
                val = '60-month loan' if val == 1 else '36-month loan'
            lines.append(f'- {label}: {val}')
    return '\n'.join(lines)

# Test
print(serialize_profile(df_200.iloc[0]))

In [ ]:
import re

def strip_think_tags(content: str) -> str:
    """Strip <think>...</think> chain-of-thought emitted by Qwen3-Max."""
    match = re.search(r'</think>\s*(.*)', content, re.DOTALL)
    return match.group(1).strip() if match else content.strip()

def parse_llm_decision(response_text: str):
    """Parse DEFAULT / NO DEFAULT from Qwen response. Returns 1, 0, or None."""
    text = strip_think_tags(response_text).upper()
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    first_line = lines[0] if lines else ''
    if 'NO DEFAULT' in first_line:
        return 0
    if 'DEFAULT' in first_line:
        return 1
    # Fallback: search entire text
    if 'NO DEFAULT' in text:
        return 0
    if 'DEFAULT' in text:
        return 1
    return None

# Verify parser handles <think> tags correctly
assert parse_llm_decision('<think>let me think</think>\nDEFAULT\nHigh DTI.') == 1
assert parse_llm_decision('NO DEFAULT\nStable income.') == 0
print('parse_llm_decision OK (with think-tag stripping)')


In [ ]:
# Load Qwen correction cache (produced by scripts/run_qwen_correction.py)
if not os.path.exists(QWEN_CACHE_PATH):
    raise FileNotFoundError(
        f'Qwen cache not found: {QWEN_CACHE_PATH}\n'
        'Run scripts/run_qwen_correction.py first, then upload the JSON to Drive.'
    )

with open(QWEN_CACHE_PATH) as f:
    qwen_cache = json.load(f)

# Populate llm_decisions and llm_reasonings dicts (same format as the old API loop)
borderline_idx = df_200[df_200['is_borderline']].index.tolist()
llm_decisions  = {}
llm_reasonings = {}
parse_errors   = 0

for row_idx in borderline_idx:
    entry = qwen_cache.get(str(row_idx), {})
    raw   = entry.get('raw', '') or entry.get('decision', '') or ''

    # Support both cache formats:
    # - run_qwen_correction.py: {decision: 'DEFAULT'/'NO DEFAULT', reasoning: str, raw: str}
    # - old format: {decision: 0/1, reasoning: str}
    if isinstance(entry.get('decision'), int):
        decision = entry['decision']
    elif isinstance(entry.get('decision'), str):
        decision = parse_llm_decision(entry['decision'])
    else:
        decision = parse_llm_decision(raw)

    if decision is None:
        decision = int(df_200.loc[row_idx, 'ml_proba'] >= 0.5)  # fallback to ML
        parse_errors += 1

    llm_decisions[row_idx]  = decision
    llm_reasonings[row_idx] = entry.get('reasoning', '')

print(f'Cache loaded: {len(qwen_cache)} entries')
print(f'Borderline cases: {len(borderline_idx)} | parse errors: {parse_errors}')
print(f'LLM decisions - DEFAULT={sum(v==1 for v in llm_decisions.values())} | '
      f'NO DEFAULT={sum(v==0 for v in llm_decisions.values())}')


## 4. Build Combined Predictions

In [ ]:
# Hybrid prediction:
# - Borderline: use LLM decision
# - Confident:  use ML decision
hybrid_pred = ml_pred_200.copy()
for row_idx, decision in llm_decisions.items():
    loc = df_200.index.get_loc(row_idx)
    hybrid_pred[loc] = decision

# Also track: how often did LLM flip the ML decision?
llm_flips = {}
for row_idx, llm_dec in llm_decisions.items():
    loc = df_200.index.get_loc(row_idx)
    ml_dec = ml_pred_200[loc]
    llm_flips[row_idx] = (ml_dec != llm_dec)

n_flips = sum(llm_flips.values())
print(f'LLM flipped ML decision on {n_flips}/{len(llm_decisions)} borderline cases ({n_flips/len(llm_decisions)*100:.1f}%)')

## 5. Evaluation

In [ ]:
def eval_predictions(y_true, y_pred, name):
    acc  = accuracy_score(y_true, y_pred)
    bacc = balanced_accuracy_score(y_true, y_pred)
    f1   = f1_score(y_true, y_pred, zero_division=0)
    return {'Method': name, 'Accuracy': round(acc, 4),
            'BACC': round(bacc, 4), 'F1': round(f1, 4)}

# Overall (200 samples)
res_ml     = eval_predictions(y_200, ml_pred_200, 'ML alone')
res_hybrid = eval_predictions(y_200, hybrid_pred,  'ML + LLM correction')

# Borderline only
res_ml_border     = eval_predictions(y_200[mask_border], ml_pred_200[mask_border], 'ML alone (borderline)')
res_hybrid_border = eval_predictions(y_200[mask_border], hybrid_pred[mask_border],  'ML + LLM (borderline)')

# Confident only
res_ml_conf = eval_predictions(y_200[~mask_border], ml_pred_200[~mask_border], 'ML alone (confident)')

results_p3 = pd.DataFrame([
    res_ml, res_hybrid,
    res_ml_border, res_hybrid_border,
    res_ml_conf,
]).set_index('Method')

print('=== PART 3 RESULTS: LLM Correction Experiment ===')
print(results_p3.to_string())

results_p3.to_csv(f'{MODEL_DIR}/part3_correction_results.csv')
print('\nSaved to Drive.')

In [ ]:
# Detailed breakdown: ML right/wrong × LLM right/wrong on borderline cases
border_df = df_200[mask_border].copy().reset_index(drop=True)
border_ml  = ml_pred_200[mask_border]
border_hyb = hybrid_pred[mask_border]
border_y   = y_200[mask_border].values

border_df['y_true']    = border_y
border_df['ml_pred']   = border_ml
border_df['llm_pred']  = border_hyb
border_df['ml_correct']  = (border_ml == border_y).astype(int)
border_df['llm_correct'] = (border_hyb == border_y).astype(int)
border_df['was_flipped'] = (border_ml != border_hyb).astype(int)

flip_outcomes = border_df[border_df['was_flipped'] == 1]
print(f'\n=== Flip Analysis (LLM changed ML decision) ===')
print(f'Total flips: {len(flip_outcomes)}')
print(f'Flips that were CORRECT (LLM improved):   {(flip_outcomes["llm_correct"] == 1).sum()}')
print(f'Flips that were WRONG   (LLM hurt):       {(flip_outcomes["llm_correct"] == 0).sum()}')

In [ ]:
# Visualization: accuracy comparison bar chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Overall vs Borderline comparison
ax = axes[0]
categories = ['Overall (200)', 'Borderline (100)', 'Confident (100)']
ml_accs  = [res_ml['Accuracy'],     res_ml_border['Accuracy'],  res_ml_conf['Accuracy']]
hyb_accs = [res_hybrid['Accuracy'], res_hybrid_border['Accuracy'], res_ml_conf['Accuracy']]

x = np.arange(len(categories))
w = 0.35
ax.bar(x - w/2, ml_accs,  w, label='ML alone',          color=COLORS['accent'],  alpha=0.85)
ax.bar(x + w/2, hyb_accs, w, label='ML + LLM correction', color=COLORS['qwen'], alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(categories)
ax.set_ylabel('Accuracy')
ax.set_title('Accuracy: ML alone vs ML + LLM Correction', fontweight='bold')
ax.legend()
ax.set_ylim(0, 1)
ax.grid(axis='y', alpha=0.3)
for i, (v1, v2) in enumerate(zip(ml_accs, hyb_accs)):
    ax.text(i - w/2, v1 + 0.01, f'{v1:.3f}', ha='center', fontsize=9)
    ax.text(i + w/2, v2 + 0.01, f'{v2:.3f}', ha='center', fontsize=9)

# Right: Flip outcome breakdown (pie)
ax = axes[1]
n_correct_flips = (flip_outcomes['llm_correct'] == 1).sum()
n_wrong_flips   = (flip_outcomes['llm_correct'] == 0).sum()
n_no_flip       = (border_df['was_flipped'] == 0).sum()
sizes  = [n_correct_flips, n_wrong_flips, n_no_flip]
labels = [
    f'LLM improved ({n_correct_flips})',
    f'LLM hurt ({n_wrong_flips})',
    f'No flip ({n_no_flip})'
]
colors = [COLORS['good'], COLORS['bad'], COLORS['neutral']]
ax.pie(sizes, labels=labels, colors=colors, autopct='%1.0f%%', startangle=90)
ax.set_title('LLM Flip Outcomes (Borderline Cases)', fontweight='bold')

plt.tight_layout()
plt.savefig(f'{DRIVE_DIR}/figures/part3_correction_results.png', dpi=150)
plt.show()

In [ ]:
# Show example cases: LLM corrected ML error
correct_flips = flip_outcomes[flip_outcomes['llm_correct'] == 1]
print(f'=== Cases where LLM corrected ML ({len(correct_flips)} cases) ===')
if len(correct_flips) > 0:
    show_cols = [c for c in ['ml_proba', 'y_true', 'ml_pred', 'llm_pred',
                              'loan_amnt', 'annual_inc', 'dti', 'fico_score', 'purpose']
                 if c in correct_flips.columns]
    print(correct_flips[show_cols].head(5).to_string())

print()
wrong_flips = flip_outcomes[flip_outcomes['llm_correct'] == 0]
print(f'=== Cases where LLM hurt ML ({len(wrong_flips)} cases) ===')
if len(wrong_flips) > 0:
    print(wrong_flips[show_cols].head(5).to_string())

In [ ]:
# Summary narrative
auc_improvement  = res_hybrid['Accuracy'] - res_ml['Accuracy']
bacc_improvement = res_hybrid['BACC']     - res_ml['BACC']
border_acc_delta = res_hybrid_border['Accuracy'] - res_ml_border['Accuracy']

print('=== SUMMARY ===')
print(f'Overall accuracy change:    {auc_improvement:+.4f}')
print(f'Overall BACC change:        {bacc_improvement:+.4f}')
print(f'Borderline accuracy change: {border_acc_delta:+.4f}')
print()
print('Interpretation:')
if border_acc_delta > 0:
    print(f'  LLM correction improved borderline accuracy by {border_acc_delta*100:.1f}pp')
    print('  → Supports the hypothesis that LLM provides complementary value on uncertain cases')
else:
    print('  LLM correction did not improve borderline accuracy in this sample')
    print('  → Consider larger sample or different prompt design')